# ColdSite-DTI — the whole analysis, once the grids have trained (Colab, T4)

Runs `python -m src.evaluation.run_all`: faithfulness and the ladder for every audited
model and seed, the Holm-corrected audit table, the non-kinase control (with and without
cotransport ions), the positive control, and one summary page. Outputs go to **Drive**,
so if Colab disconnects, run the notebook again from the top: finished steps are skipped.

## Before you run

1. **Runtime -> Change runtime type -> T4 GPU.**
2. Put each grid's output in Drive, unzipped, one folder per source, for example
   - `MyDrive/coldsite-grid36-results/` — account 1's `grid36_results.zip` contents
   - `MyDrive/coldsite-moltrans-davis-results/` — account 2's results zip contents
3. Set the settings cell below, then run the cells in order. Section 2 asks you to
   authorise Drive — that click is yours.

Nothing here trains a model. Expect roughly an hour per dataset on a T4.


## 1. Settings

In [ ]:
DATASET = 'davis'        # 'davis' | 'kiba'

# Drive folders holding trained checkpoints (.pt) and their *_results.json -- every
# model's, from every account. Files are merged into one local folder; the first copy
# of a name wins, and volume-control files (_trainsub) are never merged in.
RESULT_FOLDERS = [
    '/content/drive/MyDrive/coldsite-grid36-results',
    '/content/drive/MyDrive/coldsite-moltrans-davis-results',
]
VOLUME_CONTROL_DIR = '/content/drive/MyDrive/coldsite-volume-control'   # DAVIS only; None to skip
OUT = f'/content/drive/MyDrive/coldsite-analysis-{DATASET}'


## 2. GPU and Drive

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU  :', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)


## 3. Clone the repo

In [ ]:
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
SRC = '/content/ColdSite-DTI_New'
if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt
assert os.path.exists('src/evaluation/run_all.py'), 'This checkout predates run_all -- re-run this cell.'
!git log --oneline -1


## 4. Data and splits — verified against the rest of the project

The analysis must read the same splits every model was trained on.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.build_splits 2>&1 | grep -E 'davis|kiba|leakage'

import pandas as pd
EXPECTED = {
    'davis': {'random': (21039, 3006, 6011), 'cold_drug': (21658, 2652, 5746),
              'cold_target': (21080, 2992, 5984), 'cold_pair': (15190, 264, 1144)},
    'kiba': {'random': (82778, 11825, 23651), 'cold_drug': (83807, 12073, 22374),
             'cold_target': (85452, 10701, 22101), 'cold_pair': (58041, 1334, 4375)},
}
for split, expected in EXPECTED[DATASET].items():
    got = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{split}/{part}.csv'))
                for part in ('train', 'valid', 'test'))
    assert got == expected, f'{split}: expected {expected}, got {got}'
print(f'{DATASET} splits match.')


## 5. Gather the trained cells

Only checkpoints and their results files are copied. A volume-control file is
refused outright: `train.py` names a cut-down run exactly like a full one, so merged in
it would pass for a real `random` cell.

In [ ]:
import glob, shutil
RESULTS = '/content/results'
os.makedirs(RESULTS, exist_ok=True)
copied = kept = refused = 0
for folder in RESULT_FOLDERS:
    if not os.path.isdir(folder):
        print(f'not found, skipped: {folder}')
        continue
    for path in glob.glob(f'{folder}/**/*', recursive=True):
        name = os.path.basename(path)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        if '_trainsub' in name:
            refused += 1
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            kept += 1
            continue
        shutil.copy2(path, dst)
        copied += 1
print(f'copied {copied}, already present {kept}, refused {refused} volume-control file(s)')


## 6. What is there, and what will run (changes nothing)

In [ ]:
VC = f'--volume-control-dir {VOLUME_CONTROL_DIR}' if VOLUME_CONTROL_DIR and DATASET == 'davis' else ''
!python -m src.evaluation.run_all --dataset {DATASET} --checkpoint-dir {RESULTS} --out-dir {OUT} --dry-run


## 7. Run it

Streams every runner's output. Re-running skips whatever already finished.

In [ ]:
!python -m src.evaluation.run_all --dataset {DATASET} --checkpoint-dir {RESULTS} --out-dir {OUT} {VC}


## 8. The summary

In [ ]:
from IPython.display import Markdown, display
page = f'{OUT}/analysis_summary_{DATASET}.md'
display(Markdown(open(page).read()) if os.path.exists(page) else f'no summary yet at {page}')
